# Bootstrap legacy: minimal S3 example

The smallest end-to-end version of the `99_Bootstrap_legacy.ipynb` pattern: create a dataframe, read it through `DataHelper` (legacy `field_map` + `sticky_filters`), write it to S3, and read it back.


In [1]:
import os
import tempfile
from pathlib import Path

from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti.core.filesystem import add_endpoint_to_allowlist
from boti_data import DataHelper, ParquetReader, ParquetSink
from boti_data.connection_catalog import S3Catalog

load_dotenv(".env.local")
add_endpoint_to_allowlist(os.getenv("ETL_ENDPOINT_ALLOWLIST"))


## 1. Create a dataframe

Seed a tiny legacy-shaped SQLite table — the source `DataHelper` will read from.

In [2]:
class Base(DeclarativeBase):
    pass


class TrackingProduct(Base):
    __tablename__ = "asm_tracking_productos"

    id: Mapped[int] = mapped_column(primary_key=True)
    id_producto: Mapped[int]
    cliente_id: Mapped[int]
    id_track_global: Mapped[int]
    id_tipo_producto: Mapped[int]


db_path = Path(tempfile.mkdtemp()) / "bootstrap_legacy_minimal.db"
sqlite_dsn = f"sqlite:///{db_path}"

engine = create_engine(sqlite_dsn)
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all(
        [
            TrackingProduct(id_producto=101, cliente_id=11, id_track_global=1, id_tipo_producto=1),
            TrackingProduct(id_producto=102, cliente_id=12, id_track_global=2, id_tipo_producto=1),
            TrackingProduct(id_producto=103, cliente_id=13, id_track_global=3, id_tipo_producto=1),
        ]
    )
    session.commit()
engine.dispose()


## 2. Read data

`DataHelper` with `field_map` (legacy column names → modern) and `sticky_filters` (always-applied filter), same as `99_Bootstrap_legacy.ipynb`.

In [3]:
os.environ["BOOTSTRAP_LEGACY_MINIMAL_DSN"] = sqlite_dsn

helper = DataHelper(
    backend="sqlalchemy",
    connection_url=sqlite_dsn,
    worker_connection_env_var="BOOTSTRAP_LEGACY_MINIMAL_DSN",
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="asm_tracking_productos",
    field_map={"id_track_global": "global_track_id", "id_tipo_producto": "product_type_id"},
    sticky_filters={"product_type_id": 1},
)
frame = helper.pandas.load(columns=["id_producto", "cliente_id", "product_type_id", "global_track_id"])
frame


,id_producto,cliente_id,product_type_id,global_track_id
0,101,11,1,1
1,102,12,1,2
2,103,13,1,3


## 3. Write to S3

In [4]:
store = S3Catalog("ETL_", env_file=".env.local")
s3_path = f"{store.storage_path}/scratch/bootstrap_legacy_minimal"

with ParquetReader({"storage_path": s3_path}, fs=store.fs()) as destination:
    with ParquetSink(destination, partition_on=None) as sink:
        sink.write(frame)


## 4. Read from S3

In [5]:
with ParquetReader({"storage_path": s3_path}, fs=store.fs()) as reader:
    reloaded = reader.load(return_type="pandas")
reloaded


,id_producto,cliente_id,product_type_id,global_track_id
0,101,11,1,1
1,102,12,1,2
2,103,13,1,3


## Cleanup

In [6]:
store.fs().rm(s3_path, recursive=True)
helper.close()
